In [1]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
    average_precision_score,
    accuracy_score
)
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedShuffleSplit,
    GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from scikeras.wrappers import KerasClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# =====================================
# 0. 基本設定
# =====================================
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

# =====================================
# 1. 讀資料
# =====================================
df = pd.read_csv("final_dataset_for_ml_FULL.csv", encoding="utf-8-sig")

# =====================================
# 2. target
# =====================================
y = df["label"].astype(int)

# =====================================
# 3. feature groups (LLaMA version)
# =====================================
semantic_cols = [
    "llama_specificity_score_1",
    "llama_evidence_substantiation_score_1",
    "llama_vagueness_score_1",
    "llama_commitment_score_1",
    "llama_temporal_credibility_score_1",
    "llama_deflection_score_1",
    "llama_comparability_score_1"
]

lexical_cols = [
    "llama_has_scope_1",
    "llama_has_sbti_1",
    "llama_has_material_1",
    "llama_has_kpi_1",
    "llama_has_percent_1"
    
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

# =====================================
# 4. 檢查欄位
# =====================================
required_cols = semantic_cols + lexical_cols + financial_cols + ["label"]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# =====================================
# 5. Ablation sets (M1-M6)
# =====================================
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M2: Lexical": lexical_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}

# =====================================
# 6. class_weight
# =====================================
classes = np.unique(y)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
class_weight = dict(zip(classes, weights))

print("Class distribution:", y.value_counts().to_dict())
print("Class weight:", class_weight)

# =====================================
# 7. CV 設定
# =====================================
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
grid_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

# =====================================
# 8. 自訂 transformer：2D -> 3D for LSTM
# =====================================
class ReshapeToLSTMInput(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = np.asarray(X).astype(np.float32)
        return X.reshape((X.shape[0], X.shape[1], 1))

# =====================================
# 9. 建立前處理器
#    financial 做 scaling
#    semantic / lexical 保留原樣
# =====================================
def build_preprocessor(selected_cols):
    sem_in_use = [c for c in semantic_cols if c in selected_cols]
    lex_in_use = [c for c in lexical_cols if c in selected_cols]
    fin_in_use = [c for c in financial_cols if c in selected_cols]

    transformers = []

    if sem_in_use:
        transformers.append(
            ("semantic", Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]), sem_in_use)
        )

    if lex_in_use:
        transformers.append(
            ("lexical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent"))
            ]), lex_in_use)
        )

    if fin_in_use:
        transformers.append(
            ("financial", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), fin_in_use)
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    return preprocessor

# =====================================
# 10. 建立 LSTM 模型
# =====================================
def build_lstm_model(meta, units=16, dropout_rate=0.2, learning_rate=0.001):
    n_timesteps = meta["n_features_in_"]

    model = Sequential([
        Input(shape=(n_timesteps, 1)),
        LSTM(units=units),
        Dropout(dropout_rate),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc")
        ],
        weighted_metrics=[]
    )
    return model

# =====================================
# 11. callback 工具
# =====================================
def get_callbacks():
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5
    )

    return [early_stopping, reduce_lr]

# =====================================
# 12. 建立 LSTM pipeline
# =====================================
def build_lstm_pipeline(selected_cols):
    preprocessor = build_preprocessor(selected_cols)

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("reshape", ReshapeToLSTMInput()),
        ("classifier", KerasClassifier(
            model=build_lstm_model,
            verbose=0,
            class_weight=class_weight,
            validation_split=0.1,
            callbacks=get_callbacks(),
            random_state=SEED
        ))
    ])
    return pipeline

# =====================================
# 13. GridSearchCV 參數
# =====================================
param_grid = {
    "classifier__model__units": [16],
    "classifier__model__dropout_rate": [0.2],
    "classifier__model__learning_rate": [0.001],
    "classifier__batch_size": [8, 16],
    "classifier__epochs": [30]
}

# =====================================
# 14. 找最佳 threshold
# =====================================
def find_best_threshold(y_true, y_prob):
    best_f1 = -1
    best_th = 0.5

    for th in np.arange(0.10, 0.91, 0.05):
        y_pred = (y_prob >= th).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)

        if score > best_f1:
            best_f1 = score
            best_th = round(float(th), 2)

    return best_th, best_f1

# =====================================
# 15. 以最佳參數建立最終 pipeline
# =====================================
def build_best_pipeline(selected_cols, best_params):
    pipeline = Pipeline([
        ("preprocess", build_preprocessor(selected_cols)),
        ("reshape", ReshapeToLSTMInput()),
        ("classifier", KerasClassifier(
            model=build_lstm_model,
            verbose=0,
            class_weight=class_weight,
            validation_split=0.1,
            callbacks=get_callbacks(),
            random_state=SEED,
            units=best_params["classifier__model__units"],
            dropout_rate=best_params["classifier__model__dropout_rate"],
            learning_rate=best_params["classifier__model__learning_rate"],
            batch_size=best_params["classifier__batch_size"],
            epochs=best_params["classifier__epochs"]
        ))
    ])
    return pipeline

# =====================================
# 16. outer 10-fold evaluation
# =====================================
def evaluate_feature_set(X, y, feature_set_name):
    print(f"\nRunning {feature_set_name} ...")

    # Step 1. GridSearchCV 找最佳參數
    grid_search = GridSearchCV(
        estimator=build_lstm_pipeline(X.columns.tolist()),
        param_grid=param_grid,
        cv=grid_cv,
        scoring="f1",
        n_jobs=1,
        refit=True
    )

    grid_search.fit(X, y)
    best_params = grid_search.best_params_
    print(f"Best params for {feature_set_name}: {best_params}")

    # Step 2. outer 10-fold evaluation
    fold_metrics = []
    best_thresholds = []

    for fold_id, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        X_train_full = X.iloc[train_idx].copy()
        y_train_full = y.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()
        y_test = y.iloc[test_idx].copy()

        # 從 train fold 再切 validation 做 threshold tuning
        splitter = StratifiedShuffleSplit(
            n_splits=1,
            test_size=0.15,
            random_state=SEED + fold_id
        )

        train_sub_idx, val_sub_idx = next(splitter.split(X_train_full, y_train_full))

        X_train_sub = X_train_full.iloc[train_sub_idx].copy()
        y_train_sub = y_train_full.iloc[train_sub_idx].copy()
        X_val_sub = X_train_full.iloc[val_sub_idx].copy()
        y_val_sub = y_train_full.iloc[val_sub_idx].copy()

        # 建立最佳參數模型
        model = build_best_pipeline(X.columns.tolist(), best_params)

        # fit on train_sub
        model.fit(X_train_sub, y_train_sub)

        # validation probability -> 找最佳 threshold
        val_prob = model.predict_proba(X_val_sub)[:, 1]
        best_threshold, _ = find_best_threshold(y_val_sub, val_prob)
        best_thresholds.append(best_threshold)

        # 套到 outer test fold
        test_prob = model.predict_proba(X_test)[:, 1]
        y_pred = (test_prob >= best_threshold).astype(int)

        fold_result = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, test_prob),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob)
        }
        fold_metrics.append(fold_result)

        print(
            f"  Fold {fold_id:02d} | "
            f"threshold={best_threshold:.2f} | "
            f"F1={fold_result['f1']:.4f} | "
            f"ROC_AUC={fold_result['roc_auc']:.4f}"
        )

    # Step 3. 平均各 fold 指標
    metrics_df = pd.DataFrame(fold_metrics)

    return {
        "Model": "LSTM",
        "Feature_Set": feature_set_name,
        "Num_Features": X.shape[1],
        "Accuracy": np.mean(metrics_df["accuracy"]),
        "F1": np.mean(metrics_df["f1"]),
        "ROC_AUC": np.mean(metrics_df["roc_auc"]),
        "Precision": np.mean(metrics_df["precision"]),
        "Recall": np.mean(metrics_df["recall"]),
        "PR_AUC": np.mean(metrics_df["average_precision"]),
        "Mean_Best_Threshold": np.mean(best_thresholds)
    }

# =====================================
# 17. 執行全部 M1-M6
# =====================================
all_results = []

for feature_set_name, cols in feature_sets.items():
    X = df[cols].copy()
    result = evaluate_feature_set(X, y, feature_set_name)
    all_results.append(result)

# =====================================
# 18. 結果整理
# =====================================
results_df = pd.DataFrame(all_results)

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

print("\nFinal Results:")
print(results_df)

# =====================================
# 19. 輸出
# =====================================
output_file = "llama_LSTM_M1_M6_nestedCV_threshold_tuned_mean_only.csv"
results_df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"\nResults saved to: {output_file}")

Class distribution: {0: 296, 1: 32}
Class weight: {0: 0.5540540540540541, 1: 5.125}

Running M1: Semantic ...
Best params for M1: Semantic: {'classifier__batch_size': 8, 'classifier__epochs': 30, 'classifier__model__dropout_rate': 0.2, 'classifier__model__learning_rate': 0.001, 'classifier__model__units': 16}
  Fold 01 | threshold=0.80 | F1=0.5000 | ROC_AUC=0.8319
  Fold 02 | threshold=0.55 | F1=0.6667 | ROC_AUC=0.9741
  Fold 03 | threshold=0.70 | F1=0.0000 | ROC_AUC=0.8667
  Fold 04 | threshold=0.65 | F1=0.6667 | ROC_AUC=1.0000
  Fold 05 | threshold=0.70 | F1=0.6667 | ROC_AUC=0.9667
  Fold 06 | threshold=0.80 | F1=0.3333 | ROC_AUC=0.7111
  Fold 07 | threshold=0.65 | F1=0.4444 | ROC_AUC=0.8111
  Fold 08 | threshold=0.50 | F1=0.5455 | ROC_AUC=0.9000
  Fold 09 | threshold=0.75 | F1=0.2857 | ROC_AUC=0.9195
  Fold 10 | threshold=0.75 | F1=0.4000 | ROC_AUC=0.9540

Running M2: Lexical ...
Best params for M2: Lexical: {'classifier__batch_size': 8, 'classifier__epochs': 30, 'classifier__model_

In [2]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
    average_precision_score,
    accuracy_score
)
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedShuffleSplit,
    GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from scikeras.wrappers import KerasClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# =====================================
# 0. 基本設定
# =====================================
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

# =====================================
# 1. 讀資料
# =====================================
df = pd.read_csv("final_dataset_for_ml_FULL.csv", encoding="utf-8-sig")

# =====================================
# 2. target
# =====================================
y = df["label"].astype(int)

# =====================================
# 3. feature groups (LLaMA version)
# =====================================
semantic_cols = [
    "chatgpt_specificity_score_1",
    "chatgpt_evidence_substantiation_score_1",
    "chatgpt_vagueness_score_1",
    "chatgpt_commitment_score_1",
    "chatgpt_temporal_credibility_score_1",
    "chatgpt_deflection_score_1",
    "chatgpt_comparability_score_1"
]

lexical_cols = [
    "chatgpt_has_scope_1",
    "chatgpt_has_sbti_1",
    "chatgpt_has_material_1",
    "chatgpt_has_kpi_1",
    "chatgpt_has_percent_1"
    
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

# =====================================
# 4. 檢查欄位
# =====================================
required_cols = semantic_cols + lexical_cols + financial_cols + ["label"]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# =====================================
# 5. Ablation sets (M1-M6)
# =====================================
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M2: Lexical": lexical_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}

# =====================================
# 6. class_weight
# =====================================
classes = np.unique(y)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
class_weight = dict(zip(classes, weights))

print("Class distribution:", y.value_counts().to_dict())
print("Class weight:", class_weight)

# =====================================
# 7. CV 設定
# =====================================
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
grid_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

# =====================================
# 8. 自訂 transformer：2D -> 3D for LSTM
# =====================================
class ReshapeToLSTMInput(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = np.asarray(X).astype(np.float32)
        return X.reshape((X.shape[0], X.shape[1], 1))

# =====================================
# 9. 建立前處理器
#    financial 做 scaling
#    semantic / lexical 保留原樣
# =====================================
def build_preprocessor(selected_cols):
    sem_in_use = [c for c in semantic_cols if c in selected_cols]
    lex_in_use = [c for c in lexical_cols if c in selected_cols]
    fin_in_use = [c for c in financial_cols if c in selected_cols]

    transformers = []

    if sem_in_use:
        transformers.append(
            ("semantic", Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]), sem_in_use)
        )

    if lex_in_use:
        transformers.append(
            ("lexical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent"))
            ]), lex_in_use)
        )

    if fin_in_use:
        transformers.append(
            ("financial", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), fin_in_use)
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    return preprocessor

# =====================================
# 10. 建立 LSTM 模型
# =====================================
def build_lstm_model(meta, units=16, dropout_rate=0.2, learning_rate=0.001):
    n_timesteps = meta["n_features_in_"]

    model = Sequential([
        Input(shape=(n_timesteps, 1)),
        LSTM(units=units),
        Dropout(dropout_rate),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc")
        ],
        weighted_metrics=[]
    )
    return model

# =====================================
# 11. callback 工具
# =====================================
def get_callbacks():
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5
    )

    return [early_stopping, reduce_lr]

# =====================================
# 12. 建立 LSTM pipeline
# =====================================
def build_lstm_pipeline(selected_cols):
    preprocessor = build_preprocessor(selected_cols)

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("reshape", ReshapeToLSTMInput()),
        ("classifier", KerasClassifier(
            model=build_lstm_model,
            verbose=0,
            class_weight=class_weight,
            validation_split=0.1,
            callbacks=get_callbacks(),
            random_state=SEED
        ))
    ])
    return pipeline

# =====================================
# 13. GridSearchCV 參數
# =====================================
param_grid = {
    "classifier__model__units": [16],
    "classifier__model__dropout_rate": [0.2],
    "classifier__model__learning_rate": [0.001],
    "classifier__batch_size": [8, 16],
    "classifier__epochs": [30]
}

# =====================================
# 14. 找最佳 threshold
# =====================================
def find_best_threshold(y_true, y_prob):
    best_f1 = -1
    best_th = 0.5

    for th in np.arange(0.10, 0.91, 0.05):
        y_pred = (y_prob >= th).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)

        if score > best_f1:
            best_f1 = score
            best_th = round(float(th), 2)

    return best_th, best_f1

# =====================================
# 15. 以最佳參數建立最終 pipeline
# =====================================
def build_best_pipeline(selected_cols, best_params):
    pipeline = Pipeline([
        ("preprocess", build_preprocessor(selected_cols)),
        ("reshape", ReshapeToLSTMInput()),
        ("classifier", KerasClassifier(
            model=build_lstm_model,
            verbose=0,
            class_weight=class_weight,
            validation_split=0.1,
            callbacks=get_callbacks(),
            random_state=SEED,
            units=best_params["classifier__model__units"],
            dropout_rate=best_params["classifier__model__dropout_rate"],
            learning_rate=best_params["classifier__model__learning_rate"],
            batch_size=best_params["classifier__batch_size"],
            epochs=best_params["classifier__epochs"]
        ))
    ])
    return pipeline

# =====================================
# 16. outer 10-fold evaluation
# =====================================
def evaluate_feature_set(X, y, feature_set_name):
    print(f"\nRunning {feature_set_name} ...")

    # Step 1. GridSearchCV 找最佳參數
    grid_search = GridSearchCV(
        estimator=build_lstm_pipeline(X.columns.tolist()),
        param_grid=param_grid,
        cv=grid_cv,
        scoring="f1",
        n_jobs=1,
        refit=True
    )

    grid_search.fit(X, y)
    best_params = grid_search.best_params_
    print(f"Best params for {feature_set_name}: {best_params}")

    # Step 2. outer 10-fold evaluation
    fold_metrics = []
    best_thresholds = []

    for fold_id, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        X_train_full = X.iloc[train_idx].copy()
        y_train_full = y.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()
        y_test = y.iloc[test_idx].copy()

        # 從 train fold 再切 validation 做 threshold tuning
        splitter = StratifiedShuffleSplit(
            n_splits=1,
            test_size=0.15,
            random_state=SEED + fold_id
        )

        train_sub_idx, val_sub_idx = next(splitter.split(X_train_full, y_train_full))

        X_train_sub = X_train_full.iloc[train_sub_idx].copy()
        y_train_sub = y_train_full.iloc[train_sub_idx].copy()
        X_val_sub = X_train_full.iloc[val_sub_idx].copy()
        y_val_sub = y_train_full.iloc[val_sub_idx].copy()

        # 建立最佳參數模型
        model = build_best_pipeline(X.columns.tolist(), best_params)

        # fit on train_sub
        model.fit(X_train_sub, y_train_sub)

        # validation probability -> 找最佳 threshold
        val_prob = model.predict_proba(X_val_sub)[:, 1]
        best_threshold, _ = find_best_threshold(y_val_sub, val_prob)
        best_thresholds.append(best_threshold)

        # 套到 outer test fold
        test_prob = model.predict_proba(X_test)[:, 1]
        y_pred = (test_prob >= best_threshold).astype(int)

        fold_result = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, test_prob),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob)
        }
        fold_metrics.append(fold_result)

        print(
            f"  Fold {fold_id:02d} | "
            f"threshold={best_threshold:.2f} | "
            f"F1={fold_result['f1']:.4f} | "
            f"ROC_AUC={fold_result['roc_auc']:.4f}"
        )

    # Step 3. 平均各 fold 指標
    metrics_df = pd.DataFrame(fold_metrics)

    return {
        "Model": "LSTM",
        "Feature_Set": feature_set_name,
        "Num_Features": X.shape[1],
        "Accuracy": np.mean(metrics_df["accuracy"]),
        "F1": np.mean(metrics_df["f1"]),
        "ROC_AUC": np.mean(metrics_df["roc_auc"]),
        "Precision": np.mean(metrics_df["precision"]),
        "Recall": np.mean(metrics_df["recall"]),
        "PR_AUC": np.mean(metrics_df["average_precision"]),
        "Mean_Best_Threshold": np.mean(best_thresholds)
    }

# =====================================
# 17. 執行全部 M1-M6
# =====================================
all_results = []

for feature_set_name, cols in feature_sets.items():
    X = df[cols].copy()
    result = evaluate_feature_set(X, y, feature_set_name)
    all_results.append(result)

# =====================================
# 18. 結果整理
# =====================================
results_df = pd.DataFrame(all_results)

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

print("\nFinal Results:")
print(results_df)

# =====================================
# 19. 輸出
# =====================================
output_file = "chatgpt_LSTM_M1_M6_nestedCV_threshold_tuned_mean_only.csv"
results_df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"\nResults saved to: {output_file}")

Class distribution: {0: 296, 1: 32}
Class weight: {0: 0.5540540540540541, 1: 5.125}

Running M1: Semantic ...
Best params for M1: Semantic: {'classifier__batch_size': 8, 'classifier__epochs': 30, 'classifier__model__dropout_rate': 0.2, 'classifier__model__learning_rate': 0.001, 'classifier__model__units': 16}
  Fold 01 | threshold=0.85 | F1=0.6667 | ROC_AUC=0.8879
  Fold 02 | threshold=0.60 | F1=1.0000 | ROC_AUC=1.0000
  Fold 03 | threshold=0.35 | F1=0.8571 | ROC_AUC=0.9889
  Fold 04 | threshold=0.90 | F1=0.0000 | ROC_AUC=0.9889
  Fold 05 | threshold=0.50 | F1=0.8571 | ROC_AUC=1.0000
  Fold 06 | threshold=0.70 | F1=0.6667 | ROC_AUC=0.8778
  Fold 07 | threshold=0.50 | F1=0.4286 | ROC_AUC=0.8667
  Fold 08 | threshold=0.55 | F1=0.7500 | ROC_AUC=0.9556
  Fold 09 | threshold=0.35 | F1=0.5455 | ROC_AUC=0.9195
  Fold 10 | threshold=0.60 | F1=0.6000 | ROC_AUC=0.9540

Running M2: Lexical ...
Best params for M2: Lexical: {'classifier__batch_size': 8, 'classifier__epochs': 30, 'classifier__model_

In [1]:
import os
import random
import warnings
import json

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
    average_precision_score,
    accuracy_score
)
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedShuffleSplit,
    GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from scikeras.wrappers import KerasClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# =====================================
# 0. 基本設定
# =====================================
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")


# =====================================
# 0.1 小工具
# =====================================
def safe_metric(func, y_true, y_score_or_pred, **kwargs):
    try:
        return func(y_true, y_score_or_pred, **kwargs)
    except ValueError:
        return np.nan


def find_best_threshold(y_true, y_prob):
    best_f1 = -1
    best_th = 0.5

    for th in np.arange(0.10, 0.91, 0.05):
        y_pred = (y_prob >= th).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)

        if score > best_f1:
            best_f1 = score
            best_th = round(float(th), 2)

    return best_th, best_f1


# =====================================
# 1. 讀資料
# =====================================
df = pd.read_csv("final_dataset_for_ml_FULL.csv", encoding="utf-8-sig")


# =====================================
# 2. target
# =====================================
if "label" not in df.columns:
    raise ValueError("label 不存在於資料中")

y = df["label"].astype(int)


# =====================================
# 3. 反轉反向指標（llama version）
# 分數越高 -> 越漂綠 / 風險越高
# =====================================
reverse_map = {
    "llama_vagueness_score_1": "llama_vagueness_risk_1",
    "llama_deflection_score_1": "llama_deflection_risk_1"
}

for raw_col, risk_col in reverse_map.items():
    if raw_col not in df.columns:
        raise ValueError(f"{raw_col} 不存在於資料中，無法建立反向指標")
    df[risk_col] = 1 - df[raw_col].clip(0, 1)

print("✅ Llama 反向指標已建立")


# =====================================
# 4. feature groups (Llama version)
# 注意：semantic 改用 risk 欄位
# =====================================
semantic_cols = [
    "llama_specificity_score_1",
    "llama_evidence_substantiation_score_1",
    "llama_vagueness_risk_1",
    "llama_commitment_score_1",
    "llama_temporal_credibility_score_1",
    "llama_deflection_risk_1",
    "llama_comparability_score_1"
]

lexical_cols = [
    "llama_has_scope_1",
    "llama_has_sbti_1",
    "llama_has_material_1",
    "llama_has_kpi_1",
    "llama_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]


# =====================================
# 5. 檢查欄位
# =====================================
required_cols = semantic_cols + lexical_cols + financial_cols + ["label"]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")


# =====================================
# 6. Ablation sets (M1-M6)
# =====================================
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M2: Lexical": lexical_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}


# =====================================
# 7. class_weight
# =====================================
classes = np.unique(y)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
class_weight = dict(zip(classes, weights))

print("Class distribution:", y.value_counts().to_dict())
print("Class weight:", class_weight)


# =====================================
# 8. CV 設定
# =====================================
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
grid_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)


# =====================================
# 9. 自訂 transformer：2D -> 3D for LSTM
# =====================================
class ReshapeToLSTMInput(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = np.asarray(X).astype(np.float32)
        return X.reshape((X.shape[0], X.shape[1], 1))


# =====================================
# 10. 建立前處理器
# financial 做 scaling
# semantic / lexical 保留原樣
# =====================================
def build_preprocessor(selected_cols):
    sem_in_use = [c for c in semantic_cols if c in selected_cols]
    lex_in_use = [c for c in lexical_cols if c in selected_cols]
    fin_in_use = [c for c in financial_cols if c in selected_cols]

    transformers = []

    if sem_in_use:
        transformers.append(
            ("semantic", Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]), sem_in_use)
        )

    if lex_in_use:
        transformers.append(
            ("lexical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent"))
            ]), lex_in_use)
        )

    if fin_in_use:
        transformers.append(
            ("financial", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), fin_in_use)
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )
    return preprocessor


# =====================================
# 11. 建立 LSTM 模型
# =====================================
def build_lstm_model(meta, units=16, dropout_rate=0.2, learning_rate=0.001):
    n_timesteps = meta["n_features_in_"]

    model = Sequential([
        Input(shape=(n_timesteps, 1)),
        LSTM(units=units),
        Dropout(dropout_rate),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc")
        ],
        weighted_metrics=[]
    )
    return model


# =====================================
# 12. callback 工具
# =====================================
def get_callbacks():
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5
    )

    return [early_stopping, reduce_lr]


# =====================================
# 13. 建立 LSTM pipeline
# =====================================
def build_lstm_pipeline(selected_cols):
    preprocessor = build_preprocessor(selected_cols)

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("reshape", ReshapeToLSTMInput()),
        ("classifier", KerasClassifier(
            model=build_lstm_model,
            verbose=0,
            class_weight=class_weight,
            validation_split=0.1,
            callbacks=get_callbacks(),
            random_state=SEED
        ))
    ])
    return pipeline


# =====================================
# 14. GridSearchCV 參數
# =====================================
param_grid = {
    "classifier__model__units": [16],
    "classifier__model__dropout_rate": [0.2],
    "classifier__model__learning_rate": [0.001],
    "classifier__batch_size": [8, 16],
    "classifier__epochs": [30]
}


# =====================================
# 15. 以最佳參數建立最終 pipeline
# =====================================
def build_best_pipeline(selected_cols, best_params):
    pipeline = Pipeline([
        ("preprocess", build_preprocessor(selected_cols)),
        ("reshape", ReshapeToLSTMInput()),
        ("classifier", KerasClassifier(
            model=build_lstm_model,
            verbose=0,
            class_weight=class_weight,
            validation_split=0.1,
            callbacks=get_callbacks(),
            random_state=SEED,
            units=best_params["classifier__model__units"],
            dropout_rate=best_params["classifier__model__dropout_rate"],
            learning_rate=best_params["classifier__model__learning_rate"],
            batch_size=best_params["classifier__batch_size"],
            epochs=best_params["classifier__epochs"]
        ))
    ])
    return pipeline


# =====================================
# 16. outer 10-fold evaluation
# 回傳：summary + fold_df
# =====================================
def evaluate_feature_set(X, y, feature_set_name):
    print(f"\nRunning {feature_set_name} ...")

    # Step 1. GridSearchCV 找最佳參數
    grid_search = GridSearchCV(
        estimator=build_lstm_pipeline(X.columns.tolist()),
        param_grid=param_grid,
        cv=grid_cv,
        scoring="f1",
        n_jobs=1,
        refit=True
    )

    grid_search.fit(X, y)
    best_params = grid_search.best_params_
    print(f"Best params for {feature_set_name}: {best_params}")

    # Step 2. outer 10-fold evaluation
    fold_metrics = []
    best_thresholds = []

    for fold_id, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        X_train_full = X.iloc[train_idx].copy()
        y_train_full = y.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()
        y_test = y.iloc[test_idx].copy()

        # 從 train fold 再切 validation 做 threshold tuning
        splitter = StratifiedShuffleSplit(
            n_splits=1,
            test_size=0.15,
            random_state=SEED + fold_id
        )

        train_sub_idx, val_sub_idx = next(splitter.split(X_train_full, y_train_full))

        X_train_sub = X_train_full.iloc[train_sub_idx].copy()
        y_train_sub = y_train_full.iloc[train_sub_idx].copy()
        X_val_sub = X_train_full.iloc[val_sub_idx].copy()
        y_val_sub = y_train_full.iloc[val_sub_idx].copy()

        # 建立最佳參數模型
        model = build_best_pipeline(X.columns.tolist(), best_params)

        # fit on train_sub
        model.fit(X_train_sub, y_train_sub)

        # validation probability -> 找最佳 threshold
        val_prob = model.predict_proba(X_val_sub)[:, 1]
        best_threshold, best_val_f1 = find_best_threshold(y_val_sub, val_prob)
        best_thresholds.append(best_threshold)

        # 套到 outer test fold
        test_prob = model.predict_proba(X_test)[:, 1]
        y_pred = (test_prob >= best_threshold).astype(int)

        fold_accuracy = accuracy_score(y_test, y_pred)
        fold_f1 = f1_score(y_test, y_pred, zero_division=0)
        fold_precision = precision_score(y_test, y_pred, zero_division=0)
        fold_recall = recall_score(y_test, y_pred, zero_division=0)
        fold_roc_auc = safe_metric(roc_auc_score, y_test, test_prob)
        fold_pr_auc = safe_metric(average_precision_score, y_test, test_prob)

        fold_result = {
            "fold": fold_id,
            "Model": "LSTM",
            "Feature_Set": feature_set_name,
            "Num_Features": X.shape[1],
            "accuracy": fold_accuracy,
            "f1": fold_f1,
            "roc_auc": fold_roc_auc,
            "precision": fold_precision,
            "recall": fold_recall,
            "average_precision": fold_pr_auc,
            "best_threshold": best_threshold,
            "val_best_f1": best_val_f1,
            "best_params": json.dumps(best_params, ensure_ascii=False)
        }
        fold_metrics.append(fold_result)

        roc_text = f"{fold_roc_auc:.4f}" if pd.notna(fold_roc_auc) else "nan"
        pr_text = f"{fold_pr_auc:.4f}" if pd.notna(fold_pr_auc) else "nan"

        print(
            f"  Fold {fold_id:02d} | "
            f"threshold={best_threshold:.2f} | "
            f"F1={fold_f1:.4f} | "
            f"ROC_AUC={roc_text} | "
            f"PR_AUC={pr_text}"
        )

    # Step 3. 平均 + 標準差
    metrics_df = pd.DataFrame(fold_metrics)

    summary = {
        "Model": "LSTM",
        "Feature_Set": feature_set_name,
        "Num_Features": X.shape[1],

        "Accuracy_mean": metrics_df["accuracy"].mean(),
        "Accuracy_std": metrics_df["accuracy"].std(),

        "F1_mean": metrics_df["f1"].mean(),
        "F1_std": metrics_df["f1"].std(),

        "ROC_AUC_mean": metrics_df["roc_auc"].mean(),
        "ROC_AUC_std": metrics_df["roc_auc"].std(),

        "Precision_mean": metrics_df["precision"].mean(),
        "Precision_std": metrics_df["precision"].std(),

        "Recall_mean": metrics_df["recall"].mean(),
        "Recall_std": metrics_df["recall"].std(),

        "PR_AUC_mean": metrics_df["average_precision"].mean(),
        "PR_AUC_std": metrics_df["average_precision"].std(),

        "Mean_Best_Threshold": np.mean(best_thresholds),
        "Best_Params": json.dumps(best_params, ensure_ascii=False)
    }

    return summary, metrics_df


# =====================================
# 17. 執行全部 M1-M6
# =====================================
all_results = []
all_fold_results = []

for feature_set_name, cols in feature_sets.items():
    X = df[cols].copy()
    summary_result, fold_result_df = evaluate_feature_set(X, y, feature_set_name)

    all_results.append(summary_result)
    all_fold_results.append(fold_result_df)


# =====================================
# 18. 結果整理
# =====================================
results_df = pd.DataFrame(all_results)
folds_df = pd.concat(all_fold_results, axis=0, ignore_index=True)

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

fold_numeric_cols = folds_df.select_dtypes(include=[np.number]).columns
folds_df[fold_numeric_cols] = folds_df[fold_numeric_cols].round(4)

print("\nFinal Summary Results:")
print(results_df)

print("\nFold-level Results Preview:")
print(folds_df.head())


# =====================================
# 19. 輸出
# =====================================
summary_output_file = "llama_LSTM_M1_M6_nestedCV_threshold_tuned_mean_std.csv"
fold_output_file = "llama_LSTM_M1_M6_nestedCV_threshold_tuned_fold_results.csv"

results_df.to_csv(summary_output_file, index=False, encoding="utf-8-sig")
folds_df.to_csv(fold_output_file, index=False, encoding="utf-8-sig")

print(f"\nSummary results saved to: {summary_output_file}")
print(f"Fold-level results saved to: {fold_output_file}")

✅ Llama 反向指標已建立
Class distribution: {0: 296, 1: 32}
Class weight: {0: 0.5540540540540541, 1: 5.125}

Running M1: Semantic ...
Best params for M1: Semantic: {'classifier__batch_size': 8, 'classifier__epochs': 30, 'classifier__model__dropout_rate': 0.2, 'classifier__model__learning_rate': 0.001, 'classifier__model__units': 16}
  Fold 01 | threshold=0.75 | F1=0.5455 | ROC_AUC=0.9009 | PR_AUC=0.6436
  Fold 02 | threshold=0.65 | F1=0.7500 | ROC_AUC=0.9914 | PR_AUC=0.9500
  Fold 03 | threshold=0.85 | F1=0.0000 | ROC_AUC=0.9222 | PR_AUC=0.4667
  Fold 04 | threshold=0.55 | F1=0.4444 | ROC_AUC=0.9333 | PR_AUC=0.6429
  Fold 05 | threshold=0.70 | F1=0.6667 | ROC_AUC=0.9889 | PR_AUC=0.9167
  Fold 06 | threshold=0.85 | F1=0.0000 | ROC_AUC=0.7889 | PR_AUC=0.2611
  Fold 07 | threshold=0.60 | F1=0.6000 | ROC_AUC=0.9111 | PR_AUC=0.4762
  Fold 08 | threshold=0.85 | F1=0.8571 | ROC_AUC=1.0000 | PR_AUC=1.0000
  Fold 09 | threshold=0.45 | F1=0.6000 | ROC_AUC=0.9310 | PR_AUC=0.4778
  Fold 10 | threshold=0.8

In [2]:
import os
import random
import warnings
import json

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
    average_precision_score,
    accuracy_score
)
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedShuffleSplit,
    GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from scikeras.wrappers import KerasClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# =====================================
# 0. 基本設定
# =====================================
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")


# =====================================
# 0.1 小工具
# =====================================
def safe_metric(func, y_true, y_score_or_pred, **kwargs):
    try:
        return func(y_true, y_score_or_pred, **kwargs)
    except ValueError:
        return np.nan


def find_best_threshold(y_true, y_prob):
    best_f1 = -1
    best_th = 0.5

    for th in np.arange(0.10, 0.91, 0.05):
        y_pred = (y_prob >= th).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)

        if score > best_f1:
            best_f1 = score
            best_th = round(float(th), 2)

    return best_th, best_f1


# =====================================
# 1. 讀資料
# =====================================
df = pd.read_csv("final_dataset_for_ml_FULL.csv", encoding="utf-8-sig")


# =====================================
# 2. target
# =====================================
if "label" not in df.columns:
    raise ValueError("label 不存在於資料中")

y = df["label"].astype(int)


# =====================================
# 3. 反轉反向指標（llama version）
# 分數越高 -> 越漂綠 / 風險越高
# =====================================
reverse_map = {
    "chatgpt_vagueness_score_1": "chatgpt_vagueness_risk_1",
    "chatgpt_deflection_score_1": "chatgpt_deflection_risk_1"
}

for raw_col, risk_col in reverse_map.items():
    if raw_col not in df.columns:
        raise ValueError(f"{raw_col} 不存在於資料中，無法建立反向指標")
    df[risk_col] = 1 - df[raw_col].clip(0, 1)

print("✅ ChatGPT 反向指標已建立")


# =====================================
# 4. feature groups (ChatGPT version)
# 注意：semantic 改用 risk 欄位
# =====================================
semantic_cols = [
    "chatgpt_specificity_score_1",
    "chatgpt_evidence_substantiation_score_1",
    "chatgpt_vagueness_risk_1",
    "chatgpt_commitment_score_1",
    "chatgpt_temporal_credibility_score_1",
    "chatgpt_deflection_risk_1",
    "chatgpt_comparability_score_1"
]

lexical_cols = [
    "chatgpt_has_scope_1",
    "chatgpt_has_sbti_1",
    "chatgpt_has_material_1",
    "chatgpt_has_kpi_1",
    "chatgpt_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]


# =====================================
# 5. 檢查欄位
# =====================================
required_cols = semantic_cols + lexical_cols + financial_cols + ["label"]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")


# =====================================
# 6. Ablation sets (M1-M6)
# =====================================
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M2: Lexical": lexical_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}


# =====================================
# 7. class_weight
# =====================================
classes = np.unique(y)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
class_weight = dict(zip(classes, weights))

print("Class distribution:", y.value_counts().to_dict())
print("Class weight:", class_weight)


# =====================================
# 8. CV 設定
# =====================================
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
grid_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)


# =====================================
# 9. 自訂 transformer：2D -> 3D for LSTM
# =====================================
class ReshapeToLSTMInput(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = np.asarray(X).astype(np.float32)
        return X.reshape((X.shape[0], X.shape[1], 1))


# =====================================
# 10. 建立前處理器
# financial 做 scaling
# semantic / lexical 保留原樣
# =====================================
def build_preprocessor(selected_cols):
    sem_in_use = [c for c in semantic_cols if c in selected_cols]
    lex_in_use = [c for c in lexical_cols if c in selected_cols]
    fin_in_use = [c for c in financial_cols if c in selected_cols]

    transformers = []

    if sem_in_use:
        transformers.append(
            ("semantic", Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]), sem_in_use)
        )

    if lex_in_use:
        transformers.append(
            ("lexical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent"))
            ]), lex_in_use)
        )

    if fin_in_use:
        transformers.append(
            ("financial", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), fin_in_use)
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )
    return preprocessor


# =====================================
# 11. 建立 LSTM 模型
# =====================================
def build_lstm_model(meta, units=16, dropout_rate=0.2, learning_rate=0.001):
    n_timesteps = meta["n_features_in_"]

    model = Sequential([
        Input(shape=(n_timesteps, 1)),
        LSTM(units=units),
        Dropout(dropout_rate),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc")
        ],
        weighted_metrics=[]
    )
    return model


# =====================================
# 12. callback 工具
# =====================================
def get_callbacks():
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5
    )

    return [early_stopping, reduce_lr]


# =====================================
# 13. 建立 LSTM pipeline
# =====================================
def build_lstm_pipeline(selected_cols):
    preprocessor = build_preprocessor(selected_cols)

    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("reshape", ReshapeToLSTMInput()),
        ("classifier", KerasClassifier(
            model=build_lstm_model,
            verbose=0,
            class_weight=class_weight,
            validation_split=0.1,
            callbacks=get_callbacks(),
            random_state=SEED
        ))
    ])
    return pipeline


# =====================================
# 14. GridSearchCV 參數
# =====================================
param_grid = {
    "classifier__model__units": [16],
    "classifier__model__dropout_rate": [0.2],
    "classifier__model__learning_rate": [0.001],
    "classifier__batch_size": [8, 16],
    "classifier__epochs": [30]
}


# =====================================
# 15. 以最佳參數建立最終 pipeline
# =====================================
def build_best_pipeline(selected_cols, best_params):
    pipeline = Pipeline([
        ("preprocess", build_preprocessor(selected_cols)),
        ("reshape", ReshapeToLSTMInput()),
        ("classifier", KerasClassifier(
            model=build_lstm_model,
            verbose=0,
            class_weight=class_weight,
            validation_split=0.1,
            callbacks=get_callbacks(),
            random_state=SEED,
            units=best_params["classifier__model__units"],
            dropout_rate=best_params["classifier__model__dropout_rate"],
            learning_rate=best_params["classifier__model__learning_rate"],
            batch_size=best_params["classifier__batch_size"],
            epochs=best_params["classifier__epochs"]
        ))
    ])
    return pipeline


# =====================================
# 16. outer 10-fold evaluation
# 回傳：summary + fold_df
# =====================================
def evaluate_feature_set(X, y, feature_set_name):
    print(f"\nRunning {feature_set_name} ...")

    # Step 1. GridSearchCV 找最佳參數
    grid_search = GridSearchCV(
        estimator=build_lstm_pipeline(X.columns.tolist()),
        param_grid=param_grid,
        cv=grid_cv,
        scoring="f1",
        n_jobs=1,
        refit=True
    )

    grid_search.fit(X, y)
    best_params = grid_search.best_params_
    print(f"Best params for {feature_set_name}: {best_params}")

    # Step 2. outer 10-fold evaluation
    fold_metrics = []
    best_thresholds = []

    for fold_id, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        X_train_full = X.iloc[train_idx].copy()
        y_train_full = y.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()
        y_test = y.iloc[test_idx].copy()

        # 從 train fold 再切 validation 做 threshold tuning
        splitter = StratifiedShuffleSplit(
            n_splits=1,
            test_size=0.15,
            random_state=SEED + fold_id
        )

        train_sub_idx, val_sub_idx = next(splitter.split(X_train_full, y_train_full))

        X_train_sub = X_train_full.iloc[train_sub_idx].copy()
        y_train_sub = y_train_full.iloc[train_sub_idx].copy()
        X_val_sub = X_train_full.iloc[val_sub_idx].copy()
        y_val_sub = y_train_full.iloc[val_sub_idx].copy()

        # 建立最佳參數模型
        model = build_best_pipeline(X.columns.tolist(), best_params)

        # fit on train_sub
        model.fit(X_train_sub, y_train_sub)

        # validation probability -> 找最佳 threshold
        val_prob = model.predict_proba(X_val_sub)[:, 1]
        best_threshold, best_val_f1 = find_best_threshold(y_val_sub, val_prob)
        best_thresholds.append(best_threshold)

        # 套到 outer test fold
        test_prob = model.predict_proba(X_test)[:, 1]
        y_pred = (test_prob >= best_threshold).astype(int)

        fold_accuracy = accuracy_score(y_test, y_pred)
        fold_f1 = f1_score(y_test, y_pred, zero_division=0)
        fold_precision = precision_score(y_test, y_pred, zero_division=0)
        fold_recall = recall_score(y_test, y_pred, zero_division=0)
        fold_roc_auc = safe_metric(roc_auc_score, y_test, test_prob)
        fold_pr_auc = safe_metric(average_precision_score, y_test, test_prob)

        fold_result = {
            "fold": fold_id,
            "Model": "LSTM",
            "Feature_Set": feature_set_name,
            "Num_Features": X.shape[1],
            "accuracy": fold_accuracy,
            "f1": fold_f1,
            "roc_auc": fold_roc_auc,
            "precision": fold_precision,
            "recall": fold_recall,
            "average_precision": fold_pr_auc,
            "best_threshold": best_threshold,
            "val_best_f1": best_val_f1,
            "best_params": json.dumps(best_params, ensure_ascii=False)
        }
        fold_metrics.append(fold_result)

        roc_text = f"{fold_roc_auc:.4f}" if pd.notna(fold_roc_auc) else "nan"
        pr_text = f"{fold_pr_auc:.4f}" if pd.notna(fold_pr_auc) else "nan"

        print(
            f"  Fold {fold_id:02d} | "
            f"threshold={best_threshold:.2f} | "
            f"F1={fold_f1:.4f} | "
            f"ROC_AUC={roc_text} | "
            f"PR_AUC={pr_text}"
        )

    # Step 3. 平均 + 標準差
    metrics_df = pd.DataFrame(fold_metrics)

    summary = {
        "Model": "LSTM",
        "Feature_Set": feature_set_name,
        "Num_Features": X.shape[1],

        "Accuracy_mean": metrics_df["accuracy"].mean(),
        "Accuracy_std": metrics_df["accuracy"].std(),

        "F1_mean": metrics_df["f1"].mean(),
        "F1_std": metrics_df["f1"].std(),

        "ROC_AUC_mean": metrics_df["roc_auc"].mean(),
        "ROC_AUC_std": metrics_df["roc_auc"].std(),

        "Precision_mean": metrics_df["precision"].mean(),
        "Precision_std": metrics_df["precision"].std(),

        "Recall_mean": metrics_df["recall"].mean(),
        "Recall_std": metrics_df["recall"].std(),

        "PR_AUC_mean": metrics_df["average_precision"].mean(),
        "PR_AUC_std": metrics_df["average_precision"].std(),

        "Mean_Best_Threshold": np.mean(best_thresholds),
        "Best_Params": json.dumps(best_params, ensure_ascii=False)
    }

    return summary, metrics_df


# =====================================
# 17. 執行全部 M1-M6
# =====================================
all_results = []
all_fold_results = []

for feature_set_name, cols in feature_sets.items():
    X = df[cols].copy()
    summary_result, fold_result_df = evaluate_feature_set(X, y, feature_set_name)

    all_results.append(summary_result)
    all_fold_results.append(fold_result_df)


# =====================================
# 18. 結果整理
# =====================================
results_df = pd.DataFrame(all_results)
folds_df = pd.concat(all_fold_results, axis=0, ignore_index=True)

numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

fold_numeric_cols = folds_df.select_dtypes(include=[np.number]).columns
folds_df[fold_numeric_cols] = folds_df[fold_numeric_cols].round(4)

print("\nFinal Summary Results:")
print(results_df)

print("\nFold-level Results Preview:")
print(folds_df.head())


# =====================================
# 19. 輸出
# =====================================
summary_output_file = "chatgpt_LSTM_M1_M6_nestedCV_threshold_tuned_mean_std.csv"
fold_output_file = "chatgpt_LSTM_M1_M6_nestedCV_threshold_tuned_fold_results.csv"

results_df.to_csv(summary_output_file, index=False, encoding="utf-8-sig")
folds_df.to_csv(fold_output_file, index=False, encoding="utf-8-sig")

print(f"\nSummary results saved to: {summary_output_file}")
print(f"Fold-level results saved to: {fold_output_file}")

✅ ChatGPT 反向指標已建立
Class distribution: {0: 296, 1: 32}
Class weight: {0: 0.5540540540540541, 1: 5.125}

Running M1: Semantic ...
Best params for M1: Semantic: {'classifier__batch_size': 8, 'classifier__epochs': 30, 'classifier__model__dropout_rate': 0.2, 'classifier__model__learning_rate': 0.001, 'classifier__model__units': 16}
  Fold 01 | threshold=0.90 | F1=0.5714 | ROC_AUC=0.9353 | PR_AUC=0.5903
  Fold 02 | threshold=0.55 | F1=0.8889 | ROC_AUC=0.9914 | PR_AUC=0.9500
  Fold 03 | threshold=0.40 | F1=0.7500 | ROC_AUC=0.9889 | PR_AUC=0.9167
  Fold 04 | threshold=0.35 | F1=0.5455 | ROC_AUC=0.9778 | PR_AUC=0.8333
  Fold 05 | threshold=0.50 | F1=0.8571 | ROC_AUC=0.9889 | PR_AUC=0.9167
  Fold 06 | threshold=0.90 | F1=0.0000 | ROC_AUC=0.8667 | PR_AUC=0.3206
  Fold 07 | threshold=0.85 | F1=0.6000 | ROC_AUC=0.9444 | PR_AUC=0.5556
  Fold 08 | threshold=0.90 | F1=0.6667 | ROC_AUC=0.9667 | PR_AUC=0.6944
  Fold 09 | threshold=0.85 | F1=0.4444 | ROC_AUC=0.8851 | PR_AUC=0.4111
  Fold 10 | threshold=0